# LAFIYA 01 — N-ATLAS translation + baseline benchmark
Translates NaijaTriage-Bench into Hausa/Yoruba/Igbo **with N-ATLAS**, back-translates and filters, builds the fine-tuning sets, and scores **base N-ATLAS** (zero-shot and few-shot). Every step is resumable: if the session dies, re-run the cells.

Kaggle settings: **GPU T4 x2**, **Internet on**, Secret **HF_TOKEN** attached.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/EgwuSamuel/lafiya.git lafiya || (cd lafiya && git pull -q)
%cd lafiya

Pre-flight check: token, gated access to N-ATLaS + 4 ASR models, GPUs, chat template. Everything should say PASS before you continue.

In [ ]:
!python scripts/check_setup.py

In [ ]:
!pip install -q vllm
# --backend auto tries vLLM and falls back to transformers automatically.
# If the fallback runs out of GPU memory, restart the session and use --backend hf.

## 1. Test set first (so the baseline can start early)

In [ ]:
!python scripts/translate_natlas.py --backend auto --test-only

## 2. Training translations (anchored set, ablation set, code-switched)

In [ ]:
!python scripts/translate_natlas.py --backend auto

In [ ]:
!python scripts/build_sft.py
!cat data/translation_stats.json | head -60

## 3. Baseline: base N-ATLAS zero-shot and few-shot

In [ ]:
!python scripts/run_eval.py --backend auto --systems base base_fewshot

In [ ]:
!python scripts/compute_metrics.py

## 4. Save everything for the fine-tuning notebook

In [ ]:
!python scripts/hub_sync.py push data results